# PyTorch 訓練技巧和最佳實踐

本教程匯總了 PyTorch 深度學習訓練中的實用技巧和最佳實踐,幫助你訓練出更好的模型。

## 目錄
1. 訓練循環最佳實踐
2. 學習率調度
3. 優化器選擇
4. 模型初始化
5. 梯度裁剪
6. 混合精度訓練
7. 模型保存和加載
8. 早停和檢查點
9. 調試技巧
10. 效能優化

**作者:** AI Learning Notes  
**最後更新:** 2025-01

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import time

print(f"PyTorch 版本: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")

## 1. 訓練循環最佳實踐

標準的訓練循環應該包含以下步驟。

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """訓練一個 epoch"""
    model.train()  # 設置為訓練模式
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(tqdm(dataloader, desc="Training")):
        # 移動數據到設備
        data, target = data.to(device), target.to(device)
        
        # 清零梯度
        optimizer.zero_grad()
        
        # 前向傳播
        output = model(data)
        loss = criterion(output, target)
        
        # 反向傳播
        loss.backward()
        
        # 更新參數
        optimizer.step()
        
        # 統計
        running_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate_epoch(model, dataloader, criterion, device):
    """驗證一個 epoch"""
    model.eval()  # 設置為評估模式
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # 不計算梯度
        for data, target in tqdm(dataloader, desc="Validation"):
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            loss = criterion(output, target)
            
            running_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

## 2. 學習率調度

動態調整學習率可以提升模型性能。

In [ ]:
# 創建示例模型和優化器
model = nn.Linear(10, 5)
optimizer = optim.Adam(model.parameters(), lr=0.1)

# 1. StepLR: 每隔 step_size 個 epoch 將學習率乘以 gamma
scheduler_step = StepLR(optimizer, step_size=30, gamma=0.1)

# 2. CosineAnnealingLR: 餘弦退火
scheduler_cosine = CosineAnnealingLR(optimizer, T_max=100, eta_min=0)

# 3. ReduceLROnPlateau: 根據驗證指標動態調整
scheduler_plateau = ReduceLROnPlateau(
    optimizer, 
    mode='min',      # 監控指標是否下降
    factor=0.1,      # 新學習率 = 舊學習率 * factor
    patience=10,     # 多少個 epoch 沒改善就降低學習率
    verbose=True
)

# 4. OneCycleLR: 1cycle 策略 (fastai 推薦)
scheduler_onecycle = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.1,
    steps_per_epoch=100,
    epochs=50
)

# 可視化不同調度器
def plot_lr_schedule(scheduler, n_epochs):
    lrs = []
    optimizer = scheduler.optimizer
    for epoch in range(n_epochs):
        lrs.append(optimizer.param_groups[0]['lr'])
        scheduler.step()
    return lrs

# 比較不同調度器
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# StepLR
optimizer = optim.Adam(model.parameters(), lr=0.1)
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)
lrs = plot_lr_schedule(scheduler, 100)
axes[0, 0].plot(lrs)
axes[0, 0].set_title('StepLR')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Learning Rate')
axes[0, 0].grid(True)

# CosineAnnealingLR
optimizer = optim.Adam(model.parameters(), lr=0.1)
scheduler = CosineAnnealingLR(optimizer, T_max=100, eta_min=0)
lrs = plot_lr_schedule(scheduler, 100)
axes[0, 1].plot(lrs)
axes[0, 1].set_title('CosineAnnealingLR')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Learning Rate')
axes[0, 1].grid(True)

# ExponentialLR
optimizer = optim.Adam(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)
lrs = plot_lr_schedule(scheduler, 100)
axes[1, 0].plot(lrs)
axes[1, 0].set_title('ExponentialLR')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].grid(True)

# CosineAnnealingWarmRestarts
optimizer = optim.Adam(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2
)
lrs = plot_lr_schedule(scheduler, 100)
axes[1, 1].plot(lrs)
axes[1, 1].set_title('CosineAnnealingWarmRestarts')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print("學習率調度器使用建議:")
print("- StepLR: 簡單有效,適合大部分任務")
print("- CosineAnnealingLR: 平滑下降,常用於 ImageNet 訓練")
print("- ReduceLROnPlateau: 根據驗證集動態調整")
print("- OneCycleLR: fastai 推薦,訓練速度快")
print("- CosineAnnealingWarmRestarts: SGDR,適合長時間訓練")

## 3. 優化器選擇

不同的優化器適用於不同的任務。

In [ ]:
model = nn.Linear(10, 5)

# 1. SGD: 基礎優化器
optimizer_sgd = optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,        # 動量
    weight_decay=1e-4,   # L2 正則化
    nesterov=True        # Nesterov 動量
)

# 2. Adam: 最常用
optimizer_adam = optim.Adam(
    model.parameters(),
    lr=0.001,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0
)

# 3. AdamW: Adam + 解耦權重衰減
optimizer_adamw = optim.AdamW(
    model.parameters(),
    lr=0.001,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01    # 推薦使用 0.01
)

# 4. RMSprop: 適合 RNN
optimizer_rmsprop = optim.RMSprop(
    model.parameters(),
    lr=0.01,
    alpha=0.99,
    eps=1e-8,
    weight_decay=0
)

# 5. Adagrad: 適合稀疏資料
optimizer_adagrad = optim.Adagrad(
    model.parameters(),
    lr=0.01,
    lr_decay=0,
    weight_decay=0
)

print("優化器選擇指南:\n")
print("1. SGD + Momentum:")
print("   - 優點: 收斂穩定,泛化能力強")
print("   - 缺點: 需要仔細調整學習率")
print("   - 適用: CV 任務,尤其是大規模訓練\n")

print("2. Adam:")
print("   - 優點: 自適應學習率,容易上手")
print("   - 缺點: 可能過擬合")
print("   - 適用: NLP 任務,快速原型開發\n")

print("3. AdamW:")
print("   - 優點: 修正了 Adam 的權重衰減問題")
print("   - 缺點: 無")
print("   - 適用: Transformer 模型 (BERT、GPT等)\n")

print("4. RMSprop:")
print("   - 優點: 適合非平穩目標")
print("   - 缺點: 不如 Adam 流行")
print("   - 適用: RNN、強化學習\n")

print("推薦組合:")
print("- CNN (圖像分類): SGD + Momentum + StepLR")
print("- Transformer (NLP): AdamW + CosineAnnealingLR")
print("- RNN (序列): Adam/RMSprop + ReduceLROnPlateau")
print("- GAN: Adam (生成器) + Adam (判別器)")

## 4. 模型初始化

好的初始化可以加速收斂並提升性能。

In [ ]:
def init_weights(m):
    """模型初始化"""
    if isinstance(m, nn.Linear):
        # Xavier 初始化 (適合 tanh、sigmoid)
        nn.init.xavier_uniform_(m.weight)
        # Kaiming 初始化 (適合 ReLU)
        # nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    
    elif isinstance(m, nn.Conv2d):
        # Kaiming 初始化
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)

# 應用初始化
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 5)
)
model.apply(init_weights)

print("模型初始化完成")
print("\n初始化方法選擇:")
print("- Xavier: 適合 Sigmoid、Tanh 激活函數")
print("- Kaiming (He): 適合 ReLU 激活函數")
print("- Orthogonal: 適合 RNN")
print("- Normal/Uniform: 簡單初始化")

## 5. 梯度裁剪

防止梯度爆炸,特別是在 RNN 中。

In [ ]:
model = nn.LSTM(128, 256, num_layers=2)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 訓練循環中
x = torch.randn(32, 100, 128)
target = torch.randn(32, 100, 256)

output, _ = model(x)
loss = nn.MSELoss()(output, target)

optimizer.zero_grad()
loss.backward()

# 梯度裁剪
# 方法 1: 裁剪梯度範數
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# 方法 2: 裁剪梯度值
# torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=1.0)

optimizer.step()

print("梯度裁剪使用場景:")
print("- RNN/LSTM: 防止梯度爆炸")
print("- Transformer: 穩定訓練")
print("- GAN: 穩定對抗訓練")
print("\n推薦值:")
print("- RNN: max_norm=1.0")
print("- Transformer: max_norm=1.0")
print("- CNN: 通常不需要")

## 6. 混合精度訓練 (Automatic Mixed Precision)

使用 float16 加速訓練並節省顯存。

In [ ]:
# 只在 GPU 上使用
if torch.cuda.is_available():
    model = nn.Linear(1000, 1000).cuda()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    # 創建 GradScaler
    scaler = GradScaler()
    
    # 訓練循環
    x = torch.randn(128, 1000).cuda()
    target = torch.randn(128, 1000).cuda()
    
    optimizer.zero_grad()
    
    # 使用 autocast 進行混合精度
    with autocast(device_type='cuda'):
        output = model(x)
        loss = criterion(output, target)
    
    # 使用 scaler 進行反向傳播
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    print("混合精度訓練示例完成")
    print("\n優點:")
    print("- 訓練速度提升 1.5-3x")
    print("- 顯存使用減少約 50%")
    print("- 精度幾乎無損失")
    print("\n使用建議:")
    print("- 需要 NVIDIA GPU (Volta 架構及以上)")
    print("- 適合大模型訓練")
    print("- 某些操作會自動使用 float32 (如 softmax)")
else:
    print("混合精度訓練需要 CUDA 支持")

## 7. 模型保存和加載

正確的模型保存和加載策略。

In [ ]:
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 5)
)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 方法 1: 只保存模型參數 (推薦)
torch.save(model.state_dict(), 'model_weights.pth')

# 加載
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 5)
)
model.load_state_dict(torch.load('model_weights.pth'))
model.eval()

# 方法 2: 保存完整模型 (不推薦)
torch.save(model, 'model_complete.pth')

# 加載
model = torch.load('model_complete.pth')
model.eval()

# 方法 3: 保存訓練檢查點 (最佳實踐)
checkpoint = {
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 0.123,
    'accuracy': 95.5
}
torch.save(checkpoint, 'checkpoint.pth')

# 加載檢查點
checkpoint = torch.load('checkpoint.pth')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']

print("模型保存和加載完成")
print("\n最佳實踐:")
print("1. 使用 state_dict() 而不是整個模型")
print("2. 保存訓練檢查點包含優化器狀態")
print("3. 保存時記錄 epoch、loss 等信息")
print("4. 定期保存檢查點 (每 N 個 epoch)")
print("5. 保存最佳模型 (根據驗證集)")
print("6. 使用有意義的文件名 (包含時間戳、epoch等)")

## 8. 早停和檢查點

防止過擬合並保存最佳模型。

In [ ]:
class EarlyStopping:
    """早停機制"""
    
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth'):
        """
        Args:
            patience: 多少個 epoch 沒改善就停止
            verbose: 是否打印信息
            delta: 最小改善量
            path: 模型保存路徑
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """保存模型"""
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# 使用早停
model = nn.Linear(10, 5)
early_stopping = EarlyStopping(patience=7, verbose=True)

# 模擬訓練
for epoch in range(100):
    # 訓練...
    val_loss = np.random.random()  # 模擬驗證損失
    
    # 檢查早停
    early_stopping(val_loss, model)
    
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

# 加載最佳模型
model.load_state_dict(torch.load('checkpoint.pth'))

## 9. 調試技巧

快速定位和解決訓練問題。

In [ ]:
print("常見問題和解決方案:\n")

print("1. 損失不下降:")
print("   - 檢查學習率 (太大或太小)")
print("   - 檢查數據預處理")
print("   - 檢查損失函數是否正確")
print("   - 嘗試過擬合小批次資料 (驗證模型能力)\n")

print("2. 損失為 NaN:")
print("   - 學習率太大")
print("   - 使用梯度裁剪")
print("   - 檢查輸入數據是否包含 NaN")
print("   - 檢查激活函數 (如 log(0))\n")

print("3. 過擬合:")
print("   - 增加 Dropout")
print("   - 增加權重衰減 (weight_decay)")
print("   - 資料增強")
print("   - 減小模型容量")
print("   - 早停\n")

print("4. 欠擬合:")
print("   - 增加模型容量")
print("   - 訓練更多 epoch")
print("   - 減少正則化")
print("   - 檢查數據預處理\n")

print("5. OOM (Out of Memory):")
print("   - 減小 batch_size")
print("   - 使用混合精度訓練")
print("   - 使用梯度累積")
print("   - 減小模型大小\n")

print("調試技巧:")
print("- 先過擬合小批次資料 (驗證模型能力)")
print("- 可視化訓練曲線")
print("- 檢查梯度 (是否為 0 或過大)")
print("- 使用 TensorBoard 監控")
print("- 打印中間層輸出")
print("- 使用 torch.autograd.detect_anomaly() 檢測異常")

## 10. 效能優化

提升訓練速度的技巧。

In [ ]:
print("效能優化技巧:\n")

print("1. DataLoader 優化:")
print("   - 設置 num_workers > 0 (多進程加載)")
print("   - 設置 pin_memory=True (GPU 訓練)")
print("   - 設置 persistent_workers=True (保持 worker)")
print("   - 使用 prefetch_factor 預取資料\n")

print("2. 混合精度訓練:")
print("   - 使用 torch.cuda.amp")
print("   - 速度提升 1.5-3x")
print("   - 顯存節省約 50%\n")

print("3. 梯度累積:")
print("   - 模擬大 batch_size")
print("   - 節省顯存\n")

# 梯度累積示例
model = nn.Linear(10, 5)
optimizer = optim.Adam(model.parameters())
accumulation_steps = 4  # 累積 4 個批次

for i, (data, target) in enumerate([]):
    output = model(data)
    loss = nn.MSELoss()(output, target)
    loss = loss / accumulation_steps  # 標準化損失
    loss.backward()
    
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()

print("4. 模型並行:")
print("   - DataParallel (簡單但效率低)")
print("   - DistributedDataParallel (推薦)")
print("   - 模型並行 (超大模型)\n")

print("5. torch.compile (PyTorch 2.0+):")
print("   - model = torch.compile(model)")
print("   - 自動優化,速度提升 30-200%\n")

print("6. 其他技巧:")
print("   - 使用 inplace 操作 (如 ReLU(inplace=True))")
print("   - 避免頻繁的 CPU-GPU 傳輸")
print("   - 使用 .detach() 停止不必要的梯度計算")
print("   - 批次歸一化使用 track_running_stats=False (推理時)")
print("   - 設置 torch.backends.cudnn.benchmark=True (CNN)")

## 總結

本教程涵蓋了 PyTorch 訓練的核心技巧:

### 必知必會
1. **標準訓練循環**: train_epoch + validate_epoch
2. **學習率調度**: StepLR、CosineAnnealingLR
3. **優化器選擇**: SGD、Adam、AdamW
4. **模型初始化**: Xavier、Kaiming
5. **梯度裁剪**: 防止梯度爆炸
6. **混合精度訓練**: 加速訓練
7. **模型保存/加載**: state_dict
8. **早停機制**: 防止過擬合
9. **調試技巧**: 定位和解決問題
10. **效能優化**: 提升訓練速度

### 訓練 Checklist

**準備階段:**
- [ ] 設置隨機種子 (可重現性)
- [ ] 準備訓練/驗證/測試集
- [ ] 選擇合適的 DataLoader 參數
- [ ] 初始化模型權重

**訓練階段:**
- [ ] 選擇合適的優化器和學習率
- [ ] 使用學習率調度器
- [ ] 實現梯度裁剪 (如需要)
- [ ] 使用混合精度訓練 (如有 GPU)
- [ ] 實現早停機制
- [ ] 定期保存檢查點

**監控階段:**
- [ ] 記錄訓練/驗證損失和準確率
- [ ] 可視化訓練曲線
- [ ] 檢查梯度和權重
- [ ] 監控顯存使用

**優化階段:**
- [ ] 分析瓶頸 (資料加載/模型計算)
- [ ] 調整 batch_size
- [ ] 優化 DataLoader
- [ ] 使用梯度累積 (如需要)

### 參考資源
- [PyTorch Performance Tuning](https://pytorch.org/tutorials/recipes/recipes/tuning_guide.html)
- [PyTorch Best Practices](https://pytorch.org/tutorials/)
- [Weights & Biases](https://wandb.ai/) - 實驗管理
- [TensorBoard](https://www.tensorflow.org/tensorboard) - 可視化